# PRS association with progression

Tests whether a Parkinson's risk polygenic score predicts progression, using
the same samples, covariates and outcome as the GWAS itself.

The score comes from a PLINK2 `.sscore` file; the covariates and outcome come
from the pipeline's `prepared_data` folder, so the PCs are the ones the GWAS
actually adjusted for. The model mirrors the GWAS: logistic for `CS_disc`,
Cox for `SURV`, with the PRS standardised so the coefficient reads per
standard deviation.

In [ ]:
import glob
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from lifelines import CoxPHFitter

# The shared pipeline config lives in the repo root; point GWAS_SCRIPTS_DIR
# at it if this notebook is run from somewhere else.
SCRIPTS_DIR = Path(os.environ.get('GWAS_SCRIPTS_DIR', Path.cwd().parent))
sys.path.insert(0, str(SCRIPTS_DIR))

import gwas_config as cfg

pd.options.mode.chained_assignment = None
cfg.describe()

In [ ]:
# Mount the working bucket with gcsfuse (run once per VM).
for cmd in cfg.mount_commands():
    print(cmd)

## Parameters

In [ ]:
ANALYSIS = 'SURV'      # 'CS_disc' or 'SURV'
LABEL = 'EUR'

# PLINK2 .sscore file holding the score. Point this at whichever score set is
# being tested; SCORE1_AVG is the per-allele average, SCORE1_SUM the total.
PRS_PATH = Path(os.environ.get('GWAS_PRS_SSCORE', ''))
PRS_SCORE_COL = 'SCORE1_AVG'

N_QUANTILES = 10   # deciles, if the score is also binned
REFERENCE_Q = 5    # reference quantile for the odds ratio

run_dir = cfg.prep_path(LABEL, ANALYSIS, f'{LABEL}_{ANALYSIS}_CHROMS_1_22',
                        'prepared_data')
print(run_dir)

## Assemble the model frame

The pipeline keeps covariates, outcome and PCs in separate files; the PCs are
written per chromosome chunk, so they are concatenated and de-duplicated.

In [ ]:
def recode_pheno(s):
    """PLINK codes case/control as 1/2; models here want 0/1."""
    u = set(s.dropna().unique())
    if u <= {1, 2}:
        return s.map({1: 0, 2: 1})
    if u <= {0, 1}:
        return s
    raise ValueError(f'unexpected phenotype values: {sorted(u)}')


pc_cols = ['IID'] + [f'PC{i}' for i in range(1, 11)]
pcs = pd.concat(
    [pd.read_csv(f, sep='\t', usecols=pc_cols)
     for f in glob.glob(str(run_dir / f'{LABEL}_*_filtered.pca.harmonized.tsv'))],
    ignore_index=True).drop_duplicates(subset='IID')
print(f'{len(pcs)} samples with PCs')

In [ ]:
if ANALYSIS == 'CS_disc':
    full_df = pd.read_csv(run_dir / f'{LABEL}_case_control_filtered.pca.pheno.tsv',
                          sep='\t')
else:
    covar = pd.read_csv(run_dir / f'{LABEL}_all.tsv', sep='\t')
    y = pd.read_csv(cfg.prep_path(LABEL, ANALYSIS,
                                  cfg.pheno_file(ANALYSIS, LABEL)), sep='\t')

    full_df = covar.merge(y, on='IID', how='inner')

    # The GWAS fits each cohort as its own stratum; here it enters as dummies.
    cohort_dummies = pd.get_dummies(full_df['cohort'], prefix='cohort',
                                    drop_first=True, dtype=int)
    full_df = pd.concat([full_df, cohort_dummies], axis=1)
    full_df = full_df.drop(columns=['PHENO', 'study_type', 'cohort'])
    full_df = full_df.merge(pcs, on='IID', how='inner')

print(f'{len(full_df)} samples')
display(full_df.head())

In [ ]:
prs = pd.read_csv(PRS_PATH, sep=r'\s+')
prs.columns = prs.columns.str.lstrip('#')     # PLINK writes "#IID"
prs = prs[['IID', 'ALLELE_CT', 'NAMED_ALLELE_DOSAGE_SUM', PRS_SCORE_COL]]

df = prs.merge(full_df, on='IID', how='inner')
print(f'{len(df):,} individuals with both a score and phenotype data')
print(f'{PRS_SCORE_COL} range: {df[PRS_SCORE_COL].min():.4f} to {df[PRS_SCORE_COL].max():.4f}')

df['PRS'] = df[PRS_SCORE_COL]
df['PHENO_01'] = recode_pheno(df['y'])

# Standardise so the coefficient is per 1 SD of the score.
df['PRS_z'] = (df['PRS'] - df['PRS'].mean()) / df['PRS'].std()

## Covariates

The same adjustment as the GWAS: centred terms for the cross-sectional model,
raw baseline values plus follow-up time for survival, PCs in both, and the
cohort dummies.

In [ ]:
DURATION_COL = 'study_days'
cohort_cols = [c for c in df.columns if c.startswith('cohort')]

if ANALYSIS == 'SURV':
    COVARIATES = ['SEX', 'age_at_baseline', 'disease_duration', DURATION_COL]
else:
    COVARIATES = ['SEX_2', 'c_age', 'c_age_sq', 'c_disease_duration', 'c_dd_sq']
COVARIATES += [f'PC{i}' for i in range(1, 11)] + cohort_cols

df_model = df[['PRS', 'PRS_z', 'PHENO_01'] + COVARIATES].dropna()
print(f'{len(df_model):,} samples after dropping missing covariates '
      f'(events={int(df_model.PHENO_01.sum()):,}, '
      f'non-events={int((df_model.PHENO_01 == 0).sum()):,})')

## Fit

In [ ]:
covariate_cols = [c for c in df_model.columns
                  if c not in {'PRS', 'PRS_z', 'PHENO_01', DURATION_COL}]
covariate_cols.append('PRS_z')

if ANALYSIS == 'SURV':
    cph = CoxPHFitter()
    cph.fit(df_model, duration_col=DURATION_COL, event_col='PHENO_01',
            formula=' + '.join(covariate_cols))
    cph.print_summary()
    summary = cph.summary
else:
    X = sm.add_constant(df_model[covariate_cols].astype(float))
    y = df_model['PHENO_01'].astype(float)
    model = sm.Logit(y, X).fit(disp=False, maxiter=200)
    print(model.summary2())
    summary = model.summary2().tables[1]

In [ ]:
out_path = Path(f'prs_{ANALYSIS}_{LABEL}_model_summary.csv')
summary.to_csv(out_path)
print(f'wrote {out_path}')